# FASE 1. COMPRENSIÓN DE LOS DATOS - CALIDAD DEL AIRE (EEA)

## 1. Carga y observaciones iniciales

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("TFM_Aire_Valencia") \
    .getOrCreate()

print("Spark funcionando correctamente")
print("Versión de Spark:", spark.version)

:: loading settings :: url = jar:file:/usr/lib/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-3c4b7382-4cdf-427b-b0a5-9a7eb1713673;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.3.2 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.3.2 in central
	found org.apache.kafka#kafka-clients;2.8.1 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.8.4 in central
	found org.slf4j#slf4j-api;1.7.32 in central
	found org.apache.hadoop#hadoop-client-runtime;3.3.2 in central
	found org.spark-project.spark#unused;1.0.0 in central
	found org.apache.hadoop#hadoop-client-api;3.3.2 in central
	found commons-logging#commons-logging;1.1.3 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.apache.commons#commons-pool2;2.11.1 in central
:: resolution report 

Spark funcionando correctamente
Versión de Spark: 3.3.2


In [2]:
# Nombre de nuestro bucket
bucket = "gs://tfm_unir1"

# Ruta donde están los datos
ruta_aire = f"{bucket}/aire/eea/"

# Acceder al sistema de archivos de Google Cloud Storage
path = spark._jvm.org.apache.hadoop.fs.Path(ruta_aire)

fs = path.getFileSystem(spark._jsc.hadoopConfiguration())

# Listar archivos
archivos = fs.listStatus(path)

print("Número de elementos encontrados:", len(archivos))

for archivo in archivos[:10]:
    print(archivo.getPath().getName())

Número de elementos encontrados: 146
SP_46078004_7_8.parquet
SP_46078004_8_8.parquet
SP_46102002_7_8.parquet
SP_46102002_8_8.parquet
SP_46190005_7_8.parquet
SP_46190005_8_8.parquet
SP_46244003_7_8.parquet
SP_46244003_8_8.parquet
SP_46250030_7_8.parquet
SP_46250030_8_8.parquet


## 2. Análisis de un archivo Parquet

In [3]:
# Seleccionamos un archivo Parquet
ruta_archivo = "gs://tfm_unir1/aire/eea/SP_46078004_7_8.parquet"

# Leer el archivo con Spark
df = spark.read.parquet(ruta_archivo)

# Mostrar la estructura de las columnas
df.printSchema()

# Mostrar las primeras 5 filas
df.show(5, truncate=False)

root
 |-- Samplingpoint: string (nullable = true)
 |-- Pollutant: integer (nullable = true)
 |-- Start: timestamp (nullable = true)
 |-- End: timestamp (nullable = true)
 |-- Value: decimal(38,18) (nullable = true)
 |-- Unit: string (nullable = true)
 |-- AggType: string (nullable = true)
 |-- Validity: integer (nullable = true)
 |-- Verification: integer (nullable = true)
 |-- ResultTime: timestamp (nullable = true)
 |-- DataCapture: decimal(38,18) (nullable = true)
 |-- FkObservationLog: string (nullable = true)



+------------------+---------+-------------------+-------------------+--------------------+------+-------+--------+------------+-------------------+-----------+------------------------------------+
|Samplingpoint     |Pollutant|Start              |End                |Value               |Unit  |AggType|Validity|Verification|ResultTime         |DataCapture|FkObservationLog                    |
+------------------+---------+-------------------+-------------------+--------------------+------+-------+--------+------------+-------------------+-----------+------------------------------------+
|ES/SP_46078004_7_8|38       |2013-01-01 01:00:00|2013-01-01 02:00:00|9.000000000000000000|ug.m-3|hour   |1       |1           |2014-09-24 11:18:00|null       |b6effdd7-4257-4602-9925-f9a95ddb40c1|
|ES/SP_46078004_7_8|38       |2013-01-01 02:00:00|2013-01-01 03:00:00|8.000000000000000000|ug.m-3|hour   |1       |1           |2014-09-24 11:18:00|null       |b6effdd7-4257-4602-9925-f9a95ddb40c1|
|ES/SP_460

In [4]:
from pyspark.sql import functions as F

df.agg(
    F.count("*").alias("total_registros"),
    F.min("Start").alias("primera_medicion"),
    F.max("End").alias("ultima_medicion")).show(truncate=False)

+---------------+-------------------+-------------------+
|total_registros|primera_medicion   |ultima_medicion    |
+---------------+-------------------+-------------------+
|107468         |2013-01-01 01:00:00|2026-01-01 01:00:00|
+---------------+-------------------+-------------------+



### Distribución anual de los registros

In [5]:
from pyspark.sql import functions as F

df.groupBy(
    F.year("Start").alias("anio")
    ).agg(F.count("*").alias("registros")
    ).orderBy("anio").show(20)

+----+---------+
|anio|registros|
+----+---------+
|2013|     6713|
|2014|     7974|
|2015|     5970|
|2016|     7970|
|2017|     8712|
|2018|     8760|
|2019|     8760|
|2020|     8784|
|2021|     8760|
|2022|     8760|
|2023|     8760|
|2024|     8784|
|2025|     8760|
|2026|        1|
+----+---------+



### Evaluación incial de la calidad de los datos

In [6]:
# Comprobar si los registros contienen concentraciones de N0_2 ausentes y cómo están clasificados los indicadores de validez y verificación 
from pyspark.sql import functions as F

df.groupBy("Validity", "Verification").agg(
    F.count("*").alias("total_registros"),
    F.sum(
        F.when(F.col("Value").isNull(), 1).otherwise(0))
        .alias("valores_nulos")).orderBy("Validity", "Verification").show()

+--------+------------+---------------+-------------+
|Validity|Verification|total_registros|valores_nulos|
+--------+------------+---------------+-------------+
|      -1|           1|           4470|            0|
|       1|           1|         102998|            0|
+--------+------------+---------------+-------------+



### Identificación de periodos con mediciones inválidas

In [7]:
# veamos en qué años aparecen los registros con "Validity = -1"
from pyspark.sql import functions as F

df.groupBy(
    F.year("Start").alias("anio"),"Validity"
    ).count().orderBy("anio", "Validity").show(40)

+----+--------+-----+
|anio|Validity|count|
+----+--------+-----+
|2013|       1| 6713|
|2014|       1| 7974|
|2015|       1| 5970|
|2016|       1| 7970|
|2017|      -1|  440|
|2017|       1| 8272|
|2018|      -1|  151|
|2018|       1| 8609|
|2019|      -1|   40|
|2019|       1| 8720|
|2020|      -1| 1056|
|2020|       1| 7728|
|2021|      -1|  189|
|2021|       1| 8571|
|2022|      -1| 1471|
|2022|       1| 7289|
|2023|      -1|  133|
|2023|       1| 8627|
|2024|      -1|  501|
|2024|       1| 8283|
|2025|      -1|  489|
|2025|       1| 8271|
|2026|       1|    1|
+----+--------+-----+



In [8]:
from pyspark.sql import functions as F

df.filter(
    (F.year("Start") == 2022) &
    (F.col("Validity") == -1)
).groupBy(F.to_date("Start").alias("fecha")
).agg(F.count("*").alias("horas_afectadas")
).orderBy(F.desc("horas_afectadas"),"fecha"
).show(20, truncate=False)

+----------+---------------+
|fecha     |horas_afectadas|
+----------+---------------+
|2022-01-22|24             |
|2022-01-23|24             |
|2022-03-29|24             |
|2022-03-30|24             |
|2022-03-31|24             |
|2022-04-01|24             |
|2022-04-02|24             |
|2022-04-03|24             |
|2022-04-28|24             |
|2022-04-29|24             |
|2022-04-30|24             |
|2022-05-01|24             |
|2022-07-22|24             |
|2022-07-23|24             |
|2022-07-24|24             |
|2022-08-01|24             |
|2022-08-02|24             |
|2022-08-03|24             |
|2022-08-04|24             |
|2022-08-05|24             |
+----------+---------------+
only showing top 20 rows



In [9]:
from pyspark.sql import functions as F

df.filter(
    (F.to_date("Start") == "2022-03-29")
).select(
    "Start","Value",
    "Validity","Verification"
).orderBy("Start").show(24, truncate=False)

+-------------------+-----------------------+--------+------------+
|Start              |Value                  |Validity|Verification|
+-------------------+-----------------------+--------+------------+
|2022-03-29 00:00:00|-999.000000000000000000|-1      |1           |
|2022-03-29 01:00:00|-999.000000000000000000|-1      |1           |
|2022-03-29 02:00:00|-999.000000000000000000|-1      |1           |
|2022-03-29 03:00:00|-999.000000000000000000|-1      |1           |
|2022-03-29 04:00:00|-999.000000000000000000|-1      |1           |
|2022-03-29 05:00:00|-999.000000000000000000|-1      |1           |
|2022-03-29 06:00:00|-999.000000000000000000|-1      |1           |
|2022-03-29 07:00:00|-999.000000000000000000|-1      |1           |
|2022-03-29 08:00:00|-999.000000000000000000|-1      |1           |
|2022-03-29 09:00:00|-999.000000000000000000|-1      |1           |
|2022-03-29 10:00:00|-999.000000000000000000|-1      |1           |
|2022-03-29 11:00:00|-999.000000000000000000|-1 

In [10]:
# comprobamos si todos los registros con "Validity = -1" tienen "Value = -999"
# aunque "Value" no tenga valores NULL, puede contener valores inválidos
from pyspark.sql import functions as F

df.groupBy("Validity").agg(
    F.count("*").alias("total_registros"),
    F.sum(F.when(F.col("Value") == -999, 1).otherwise(0)
    ).alias("valores_menos_999"),
    F.min("Value").alias("valor_minimo"),
    F.max("Value").alias("valor_maximo")).orderBy("Validity").show()

+--------+---------------+-----------------+--------------------+--------------------+
|Validity|total_registros|valores_menos_999|        valor_minimo|        valor_maximo|
+--------+---------------+-----------------+--------------------+--------------------+
|      -1|           4470|             4470|-999.000000000000...|-999.000000000000...|
|       1|         102998|                0|               0E-18|424.0000000000000...|
+--------+---------------+-----------------+--------------------+--------------------+



- Todos los registros con Validity = -1 contienen Value = -999
- Todos los registros con Validity = 1 tienen valores comprendidos entre 0 y 424 mug/m^3
- No hemos encontrado valores -999 entre los registrros marcados como válidos

In [11]:
# comprobamos si existen registros duplicados
from pyspark.sql import functions as F

duplicados = (
    df.groupBy("Samplingpoint", "Start")
      .count()
      .filter(F.col("count") > 1))

print("Número de combinaciones duplicadas:",duplicados.count())

duplicados.show(10, truncate=False)

Número de combinaciones duplicadas: 0


+-------------+-----+-----+
|Samplingpoint|Start|count|
+-------------+-----+-----+
+-------------+-----+-----+



### Análisis exploratorio preliminar

In [12]:
# primeras estadísticas descriptivas
from pyspark.sql import functions as F

df_validos = df.filter(
    (F.col("Validity") == 1) & (F.col("Value") != -999))

df_validos.select("Value").summary(
    "count","mean","stddev","min",
    "25%","50%","75%","max").show()

+-------+--------------------+
|summary|               Value|
+-------+--------------------+
|  count|              102998|
|   mean|6.695188256082642...|
| stddev|   17.51028858886208|
|    min|               0E-18|
|    25%|                 1.0|
|    50%|                 2.0|
|    75%|                 5.0|
|    max|424.0000000000000...|
+-------+--------------------+



- La concentración habitual parece relativamente baja: la mediana es de 2, lo que significa que aproximadamente la mitad de las observaciones presentan concentraciones iguales o inferiores a ese valor. 
- La media es mayor que la mediana: esto indica una distribución asimétrica hacia la derecha (muchas mediciones bajas y algunas concentraciones elevadas)
- Hay valores extremos que debemos investigar ya que el máximo es 424
- Estos resultados corresponden únicamente a una estación y un contaminante. 

### Evolución anual de las concentraciones

In [13]:
from pyspark.sql import functions as F

df_validos.groupBy(
    F.year("Start").alias("anio")
).agg(
    F.round(F.avg("Value"), 2).alias("media_NO2"),
    F.round(F.max("Value"), 2).alias("maximo_NO2"),
    F.count("*").alias("registros_validos")
).orderBy("anio").show(20)

+----+---------+----------+-----------------+
|anio|media_NO2|maximo_NO2|registros_validos|
+----+---------+----------+-----------------+
|2013|     9.54|    334.00|             6713|
|2014|     7.85|    358.00|             7974|
|2015|    13.34|    360.00|             5970|
|2016|     6.17|    381.00|             7970|
|2017|     6.70|    378.00|             8272|
|2018|     6.94|    341.00|             8609|
|2019|     5.99|    424.00|             8720|
|2020|     5.98|    366.00|             7728|
|2021|     5.39|    333.00|             8571|
|2022|     7.72|    263.00|             7289|
|2023|     3.61|    135.00|             8627|
|2024|     5.11|    206.00|             8283|
|2025|     5.39|    177.00|             8271|
|2026|     3.00|      3.00|                1|
+----+---------+----------+-----------------+



### Identificación de valores extremos

In [14]:
# investigamos las 10 mediciones válidas más altas
df_validos.select(
    "Start","Value","Validity"
).orderBy(F.desc("Value")
).show(10, truncate=False)

+-------------------+----------------------+--------+
|Start              |Value                 |Validity|
+-------------------+----------------------+--------+
|2019-01-17 21:00:00|424.000000000000000000|1       |
|2016-12-22 21:00:00|381.000000000000000000|1       |
|2016-12-12 21:00:00|381.000000000000000000|1       |
|2017-11-23 10:00:00|378.000000000000000000|1       |
|2019-01-17 20:00:00|372.000000000000000000|1       |
|2020-01-08 09:00:00|366.000000000000000000|1       |
|2015-12-18 08:00:00|360.000000000000000000|1       |
|2020-01-16 08:00:00|359.000000000000000000|1       |
|2014-01-08 08:00:00|358.000000000000000000|1       |
|2016-12-22 20:00:00|357.000000000000000000|1       |
+-------------------+----------------------+--------+
only showing top 10 rows



Tres observaciones:
- Algunos máximos aparecen en horas consecutivas: por ejemplo, el 17 de enero de 2019 encontramos 372 a las 20:00 y 424 a las 21:00
- Todos los máximos que vemos se producen en invierno
- Varios coinciden con horarios de elevada actividad urbana, como las 08:00-10:00 y 20:00-21:00

### Análisis de un episodio de concentración elevada

In [15]:
from pyspark.sql import functions as F

df.filter(
    (F.col("Start") >= "2019-01-17 15:00:00") &
    (F.col("Start") <= "2019-01-18 03:00:00")
).select("Start","Value","Validity"
).orderBy("Start").show(20, truncate=False)

+-------------------+----------------------+--------+
|Start              |Value                 |Validity|
+-------------------+----------------------+--------+
|2019-01-17 15:00:00|58.000000000000000000 |1       |
|2019-01-17 16:00:00|76.000000000000000000 |1       |
|2019-01-17 17:00:00|118.000000000000000000|1       |
|2019-01-17 18:00:00|225.000000000000000000|1       |
|2019-01-17 19:00:00|303.000000000000000000|1       |
|2019-01-17 20:00:00|372.000000000000000000|1       |
|2019-01-17 21:00:00|424.000000000000000000|1       |
|2019-01-17 22:00:00|302.000000000000000000|1       |
|2019-01-17 23:00:00|138.000000000000000000|1       |
|2019-01-18 00:00:00|21.000000000000000000 |1       |
|2019-01-18 01:00:00|2.000000000000000000  |1       |
|2019-01-18 02:00:00|1.000000000000000000  |1       |
|2019-01-18 03:00:00|1.000000000000000000  |1       |
+-------------------+----------------------+--------+



Estudiamos qué ocurrió alrededor del máximo del 17 de enero para comprobar si la concentración aumentó progresivamente o si fue un pico aislado. 
Observamos varias horas consecutivas con concentraciones elevadas y una evolución temporal coherente

## 3. Caracterización del conjunto completo EEA

Una vez investigado el primer Parquet, tomamos ya el conjunto EEA

In [16]:
from pyspark.sql import functions as F

# Leer todos los Parquet EEA
ruta_eea = "gs://tfm_unir1/aire/eea/*.parquet"

df_eea = spark.read.parquet(ruta_eea)

# Identificar los contaminantes disponibles
df_eea.select("Pollutant").distinct().orderBy("Pollutant").show()

+---------+
|Pollutant|
+---------+
|        1|
|        5|
|        7|
|        8|
|        9|
|       10|
|       20|
|       21|
|       38|
|       78|
|      643|
|     5012|
|     5014|
|     5015|
|     5018|
|     5029|
|     5419|
|     5610|
|     5617|
|     5626|
+---------+
only showing top 20 rows



La selección del dióxido de nitrógeno (NO₂), las partículas en suspensión (PM10 y PM2.5) y el ozono troposférico (O₃) responde a su relevancia para la salud pública, su inclusión en las directrices de calidad del aire de la Organización Mundial de la Salud y sus diferentes mecanismos de emisión y formación. Esta selección permite abordar la contaminación urbana desde distintas perspectivas y explorar su relación con la movilidad, las condiciones ambientales y las características del entorno urbano.

https://www.who.int/publications/i/item/9789240034228

https://www.eea.europa.eu/en/analysis/publications/sustainability-of-europes-mobility-systems-2025/air-pollution

In [17]:
from pyspark.sql import functions as F

# Códigos oficiales de los contaminantes del TFM
codigos = {
    8: "NO2",
    7: "O3",
    5: "PM10",
    6001: "PM2.5"}

# Comprobar cuáles existen en nuestros datos
df_eea.filter(
    F.col("Pollutant").isin(list(codigos.keys()))
).groupBy("Pollutant").count().orderBy("Pollutant").show()

+---------+-------+
|Pollutant|  count|
+---------+-------+
|        5| 605201|
|        7|1081406|
|        8|1106507|
|     6001| 591819|
+---------+-------+



### Análisis de cobertura espacial y temporal

In [19]:
# puntos de medición que tenemos por contaminante
from pyspark.sql import functions as F

df_eea.filter(
    F.col("Pollutant").isin([8, 7, 5, 6001])
).groupBy("Pollutant").agg(
    F.countDistinct("Samplingpoint").alias("puntos_medicion"),
    F.min("Start").alias("primera_fecha"),
    F.max("Start").alias("ultima_fecha")
).orderBy("Pollutant").show(truncate=False)

+---------+---------------+-------------------+-------------------+
|Pollutant|puntos_medicion|primera_fecha      |ultima_fecha       |
+---------+---------------+-------------------+-------------------+
|5        |14             |2013-01-01 00:00:00|2026-01-01 00:00:00|
|7        |11             |2013-01-01 01:00:00|2026-01-01 00:00:00|
|8        |12             |2013-01-01 01:00:00|2026-01-01 00:00:00|
|6001     |9              |2013-01-01 00:00:00|2026-01-01 00:00:00|
+---------+---------------+-------------------+-------------------+



In [20]:
from pyspark.sql import functions as F

df_eea.filter(
    F.col("Pollutant").isin([8, 7, 5, 6001])
).groupBy("Pollutant","Samplingpoint"
).agg(
    F.min("Start").alias("inicio"),
    F.max("Start").alias("fin"),
    F.count("*").alias("registros")
).orderBy("Pollutant","Samplingpoint"
).show(50, truncate=False)

+---------+--------------------+-------------------+-------------------+---------+
|Pollutant|Samplingpoint       |inicio             |fin                |registros|
+---------+--------------------+-------------------+-------------------+---------+
|5        |ES/SP_46078004_10_M |2013-01-01 00:00:00|2025-12-31 00:00:00|4494     |
|5        |ES/SP_46102002_10_46|2013-01-01 01:00:00|2026-01-01 00:00:00|110253   |
|5        |ES/SP_46190005_10_M |2013-01-01 00:00:00|2025-12-31 00:00:00|4423     |
|5        |ES/SP_46244003_10_46|2021-12-01 01:00:00|2025-01-01 00:00:00|27024    |
|5        |ES/SP_46244003_10_M |2017-01-01 00:00:00|2017-01-17 00:00:00|16       |
|5        |ES/SP_46250030_10_46|2013-01-01 01:00:00|2026-01-01 00:00:00|104325   |
|5        |ES/SP_46250030_10_M |2018-01-25 00:00:00|2018-12-30 00:00:00|296      |
|5        |ES/SP_46250043_10_M |2013-01-01 00:00:00|2025-12-31 00:00:00|4480     |
|5        |ES/SP_46250046_10_46|2013-01-01 01:00:00|2026-01-01 00:00:00|113090   |
|5  

In [21]:
from pyspark.sql import functions as F

df_eea.filter(
    F.col("Pollutant").isin([8, 7, 5, 6001])
).groupBy("Pollutant","AggType"
).agg(
    F.count("*").alias("registros"),
    F.countDistinct("Samplingpoint").alias("puntos_medicion")
).orderBy("Pollutant","AggType"
).show(truncate=False)

+---------+-------+---------+---------------+
|Pollutant|AggType|registros|puntos_medicion|
+---------+-------+---------+---------------+
|5        |day    |18109    |7              |
|5        |hour   |587092   |7              |
|7        |hour   |1081406  |11             |
|8        |hour   |1106507  |12             |
|6001     |day    |4737     |2              |
|6001     |hour   |587082   |7              |
+---------+-------+---------+---------------+



### Evaluación de la calidad de los datos del conjunto

In [22]:
from pyspark.sql import functions as F

df_eea.filter(
    (F.col("Pollutant").isin([8, 7, 5, 6001])) &
    (F.col("AggType") == "hour")
).groupBy("Pollutant").agg(
    F.count("*").alias("total_registros"),
    F.sum(
        F.when(F.col("Validity") == 1, 1).otherwise(0)
    ).alias("registros_validos"),
    F.sum(
        F.when(F.col("Validity") != 1, 1).otherwise(0)
    ).alias("registros_no_validos"),
    F.sum(
        F.when(F.col("Value") == -999, 1).otherwise(0)
    ).alias("valores_menos_999")
).orderBy("Pollutant").show()

+---------+---------------+-----------------+--------------------+-----------------+
|Pollutant|total_registros|registros_validos|registros_no_validos|valores_menos_999|
+---------+---------------+-----------------+--------------------+-----------------+
|        5|         587092|           547904|               39188|            39188|
|        7|        1081406|           954047|              127359|            47044|
|        8|        1106507|          1028982|               77525|            77525|
|     6001|         587082|           547482|               39600|            39600|
+---------+---------------+-----------------+--------------------+-----------------+



- La mayoría de los registros están marcados como válidos
- En el ozono hay registros marcados como no válidos que no tienen el valor -999 (tienen Validity != 1). En los otros tres contaminantes, los recuentos de registros no válidos y valores -999 coinciden. 

Investigamos los indicadores de validez del ozono

In [23]:
from pyspark.sql import functions as F

df_eea.filter(
    (F.col("Pollutant") == 7) &
    (F.col("AggType") == "hour") &
    (F.col("Validity") != 1)
).groupBy("Validity").agg(
    F.count("*").alias("registros"),
    F.min("Value").alias("valor_minimo"),
    F.max("Value").alias("valor_maximo"),
    F.sum(
        F.when(F.col("Value") == -999, 1).otherwise(0)
    ).alias("valores_menos_999")
).orderBy("Validity").show(truncate=False)

+--------+---------+-----------------------+-----------------------+-----------------+
|Validity|registros|valor_minimo           |valor_maximo           |valores_menos_999|
+--------+---------+-----------------------+-----------------------+-----------------+
|-1      |47044    |-999.000000000000000000|-999.000000000000000000|47044            |
|4       |80315    |0E-18                  |160.000000000000000000 |0                |
+--------+---------+-----------------------+-----------------------+-----------------+



Hemos comprobado que "Validity = 4" no es un error, por tanto, corregimos la clasificación 

In [24]:
from pyspark.sql import functions as F

df_eea.filter(
    (F.col("Pollutant").isin([8, 7, 5, 6001])) &
    (F.col("AggType") == "hour")
).groupBy(
    "Pollutant", "Validity"
).count().orderBy(
    "Pollutant", "Validity"
).show(30)

+---------+--------+-------+
|Pollutant|Validity|  count|
+---------+--------+-------+
|        5|      -1|  39188|
|        5|       1| 547904|
|        7|      -1|  47044|
|        7|       1| 954047|
|        7|       4|  80315|
|        8|      -1|  77525|
|        8|       1|1028982|
|     6001|      -1|  39600|
|     6001|       1| 547482|
+---------+--------+-------+



In [25]:
# veamos si los registros inválidos están repartidos entre todas las estaciones o se concentran en unas pocas
from pyspark.sql import functions as F

df_eea.filter(
    (F.col("Pollutant").isin([8, 7, 5, 6001])) &
    (F.col("AggType") == "hour")
).groupBy(
    "Pollutant", "Samplingpoint"
).agg(
    F.count("*").alias("total"),
    F.sum(
        F.when(
            (F.col("Validity") == 1) |
            ((F.col("Pollutant") == 7) &
             (F.col("Validity") == 4)),
            1
        ).otherwise(0)
    ).alias("validos")
).withColumn(
    "porcentaje_validos",
    F.round(F.col("validos") / F.col("total") * 100, 2)
).orderBy(
    "Pollutant", "porcentaje_validos"
).show(50, truncate=False)

+---------+--------------------+------+-------+------------------+
|Pollutant|Samplingpoint       |total |validos|porcentaje_validos|
+---------+--------------------+------+-------+------------------+
|5        |ES/SP_46244003_10_46|27024 |11015  |40.76             |
|5        |ES/SP_46250047_10_46|78864 |70007  |88.77             |
|5        |ES/SP_46250054_10_46|43824 |41309  |94.26             |
|5        |ES/SP_46250046_10_46|113090|108969 |96.36             |
|5        |ES/SP_46250048_10_46|109712|106901 |97.44             |
|5        |ES/SP_46102002_10_46|110253|107604 |97.6              |
|5        |ES/SP_46250030_10_46|104325|102099 |97.87             |
|7        |ES/SP_46244003_14_6 |70056 |54270  |77.47             |
|7        |ES/SP_46250030_14_6 |110663|105306 |95.16             |
|7        |ES/SP_46250050_14_6 |113086|107981 |95.49             |
|7        |ES/SP_46190005_14_6 |109567|105849 |96.61             |
|7        |ES/SP_46250046_14_6 |111428|107814 |96.76          

In [26]:
from pyspark.sql import functions as F

df_eea.filter(
    (F.col("Pollutant") == 8) &
    (F.col("AggType") == "hour") &
    (F.col("Samplingpoint") == "ES/SP_46244003_8_8")
).groupBy(
    F.year("Start").alias("anio")
).agg(
    F.count("*").alias("total"),
    F.sum(
        F.when(F.col("Validity") == 1, 1).otherwise(0)
    ).alias("validos")
).withColumn(
    "porcentaje_validos",
    F.round(F.col("validos") / F.col("total") * 100, 2)
).orderBy("anio").show(20)

+----+-----+-------+------------------+
|anio|total|validos|porcentaje_validos|
+----+-----+-------+------------------+
|2017| 8711|    898|             10.31|
|2018|    1|      0|               0.0|
|2019| 8736|    506|              5.79|
|2020| 8783|   1316|             14.98|
|2021| 8760|    622|               7.1|
|2022| 8760|   6885|              78.6|
|2023| 8760|   1413|             16.13|
|2024| 8784|   2492|             28.37|
|2025|    1|      0|               0.0|
+----+-----+-------+------------------+



In [27]:
from pyspark.sql import functions as F

df_eea.filter(
    (F.col("Samplingpoint").contains("SP_46244003_")) &
    (F.col("Pollutant").isin([5, 7, 8, 6001])) &
    (F.col("AggType") == "hour")
).groupBy(F.year("Start").alias("anio"),"Pollutant"
).agg(
    F.count("*").alias("total"),
    F.sum(
        F.when(
            (F.col("Validity") == 1) |
            ((F.col("Pollutant") == 7) &
             (F.col("Validity") == 4)),1
        ).otherwise(0)
    ).alias("validos")
).withColumn(
    "porcentaje_validos",
    F.round(F.col("validos") / F.col("total") * 100, 2)
).orderBy("anio", "Pollutant").show(50)

+----+---------+-----+-------+------------------+
|anio|Pollutant|total|validos|porcentaje_validos|
+----+---------+-----+-------+------------------+
|2017|        7| 8711|    859|              9.86|
|2017|        8| 8711|    898|             10.31|
|2018|        7| 8760|   5175|             59.08|
|2018|        8|    1|      0|               0.0|
|2019|        7|    1|      0|               0.0|
|2019|        8| 8736|    506|              5.79|
|2020|        7| 8760|   8507|             97.11|
|2020|        8| 8783|   1316|             14.98|
|2021|        5|  720|    708|             98.33|
|2021|        7| 8759|   7895|             90.14|
|2021|        8| 8760|    622|               7.1|
|2021|     6001|  720|    708|             98.33|
|2022|        5| 8759|   6404|             73.11|
|2022|        7| 8760|   7482|             85.41|
|2022|        8| 8760|   6885|              78.6|
|2022|     6001| 8759|   6405|             73.12|
|2023|        5| 8760|   1396|             15.94|


Vamos a calcular la cobertura anual de observaciones válidas para todos los puntos de medición y los cuatro contaminantes. Así podremos identificar los problemas de manera sistemática, sin ir estación por estación manualmente

In [28]:
from pyspark.sql import functions as F

# Seleccionar los cuatro contaminantes y registros horarios
df_horario = df_eea.filter(
    (F.col("Pollutant").isin([5, 7, 8, 6001])) &
    (F.col("AggType") == "hour") &
    (F.year("Start").between(2013, 2025)))

# Calcular la cobertura por punto, contaminante y año
df_cobertura = (
    df_horario
    .withColumn("anio", F.year("Start"))
    .withColumn(
        "es_valido",
        F.when(
            (F.col("Validity") == 1) |
            ((F.col("Pollutant") == 7) &
             (F.col("Validity") == 4)),1).otherwise(0))
    .groupBy("Samplingpoint", "Pollutant", "anio")
    .agg(
        F.count("*").alias("registros"),
        F.countDistinct(
            F.when(F.col("es_valido") == 1, F.col("Start"))
        ).alias("horas_validas"))
    .withColumn(
        "horas_esperadas",
        F.when(
            ((F.col("anio") % 4 == 0) &
             ((F.col("anio") % 100 != 0) |
              (F.col("anio") % 400 == 0))),
            8784).otherwise(8760))
    .withColumn(
        "cobertura_pct",
        F.round(
            F.col("horas_validas") /
            F.col("horas_esperadas") * 100, 2)
    )
)

# Resumen global por contaminante y año
df_cobertura.groupBy("Pollutant", "anio").agg(
    F.round(F.avg("cobertura_pct"), 2).alias("cobertura_media_pct"),
    F.count("*").alias("puntos_con_registros")
).orderBy("Pollutant", "anio").show(60)

+---------+----+-------------------+--------------------+
|Pollutant|anio|cobertura_media_pct|puntos_con_registros|
+---------+----+-------------------+--------------------+
|        5|2013|              89.66|                   4|
|        5|2014|              94.39|                   4|
|        5|2015|              92.33|                   4|
|        5|2016|              96.58|                   4|
|        5|2017|              83.51|                   5|
|        5|2018|              78.19|                   5|
|        5|2019|              97.11|                   5|
|        5|2020|              96.22|                   5|
|        5|2021|              81.97|                   7|
|        5|2022|              92.11|                   7|
|        5|2023|              85.12|                   7|
|        5|2024|              86.48|                   7|
|        5|2025|              80.53|                   7|
|        7|2013|              90.48|                   9|
|        7|201

### Registros duplicados

In [29]:
# datos duplicados en todo el conjunto
from pyspark.sql import functions as F

# Buscar duplicados en todos los contaminantes horarios
df_duplicados = (
    df_horario
    .groupBy("Samplingpoint", "Pollutant", "Start")
    .agg(F.count("*").alias("repeticiones"))
    .filter(F.col("repeticiones") > 1))

# Resumen
df_duplicados.agg(
    F.count("*").alias("combinaciones_duplicadas"),
    F.coalesce(
        F.sum(F.col("repeticiones") - 1),
        F.lit(0)
    ).alias("registros_excedentes")
).show()

+------------------------+--------------------+
|combinaciones_duplicadas|registros_excedentes|
+------------------------+--------------------+
|                       0|                   0|
+------------------------+--------------------+



### Consistencia de los intervalos temporales

In [30]:
# consistencia de los intervalos temporales
from pyspark.sql import functions as F

df_horario.withColumn(
    "duracion_minutos",
    (F.col("End").cast("long") -
     F.col("Start").cast("long")) / 60
).groupBy("duracion_minutos").count() \
 .orderBy("duracion_minutos").show(20)

+----------------+-------+
|duracion_minutos|  count|
+----------------+-------+
|            60.0|3362055|
+----------------+-------+



### Verificación de unidades de medida

In [31]:
# unidades de medida
from pyspark.sql import functions as F

df_horario.groupBy(
    "Pollutant",
    "Unit"
).agg(F.count("*").alias("registros")
).orderBy("Pollutant","Unit"
).show(30, truncate=False)

+---------+------+---------+
|Pollutant|Unit  |registros|
+---------+------+---------+
|5        |ug.m-3|587086   |
|7        |ug.m-3|1081396  |
|8        |ug.m-3|1106497  |
|6001     |ug.m-3|587076   |
+---------+------+---------+



### Valores faltantes

In [32]:
# nulos 
from pyspark.sql import functions as F

df_horario.filter(
    F.col("Unit").isNull()
).groupBy("Pollutant").count() \
 .orderBy("Pollutant").show()

+---------+-----+
|Pollutant|count|
+---------+-----+
+---------+-----+

